In [2]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import confusion_matrix


In [3]:
package = joblib.load("deployment_package.pkl")

model = package["model"]
threshold = package["threshold"]


In [4]:
X_test = joblib.load("X_test.pkl")
y_test = joblib.load("y_test.pkl")


### Cross-Validation

In [5]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    model,
    X_test,
    y_test,
    cv=5,
    scoring="roc_auc"
)

scores


array([0.69649123, 0.82192982, 0.85526316, 0.69360902, 0.8       ])

In [6]:
scores.mean(), scores.std()


(np.float64(0.7734586466165413), np.float64(0.06640154000697782))

### Threshold Stability Test (Does the cost-optimal threshold change across different splits?)
 

In [7]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

thresholds_list = []

for train_idx, test_idx in skf.split(X_test, y_test):
    X_tr, X_te = X_test.iloc[train_idx], X_test.iloc[test_idx]
    y_tr, y_te = y_test.iloc[train_idx], y_test.iloc[test_idx]

    proba = model.predict_proba(X_te)[:,1]

    cost_vals = []
    for t in np.linspace(0.1, 0.9, 30):
        y_pred = (proba >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_te, y_pred).ravel()
        cost = 1*fp + 5*fn
        cost_vals.append((t, cost))

    best_t = min(cost_vals, key=lambda x: x[1])[0]
    thresholds_list.append(best_t)

thresholds_list


[np.float64(0.21034482758620693),
 np.float64(0.21034482758620693),
 np.float64(0.12758620689655173),
 np.float64(0.1827586206896552),
 np.float64(0.12758620689655173)]

In [8]:
np.mean(thresholds_list), np.std(thresholds_list)


(np.float64(0.17172413793103453), np.float64(0.037419751631035975))

### The cross-validation results show that the optimized cost-based threshold (mean ≈ 0.17, std ≈ 0.037) exhibits acceptable stability across different data splits. The threshold values vary within a relatively narrow range (0.13–0.21), which is expected given the limited dataset size and asymmetric cost structure. No extreme fluctuations or structural instability are observed. Therefore, the cost-sensitive decision policy can be considered robust, and the selected global threshold (≈0.17) is technically justified for deployment.